# Mamba Design

Designing Mamba architecture based on this [block diagram](https://www.researchgate.net/figure/The-Mamba-block-architecture-is-constructed-based-on-SSMs-mathematical-formulation-in_fig1_383792530). 

(1) SSM Equation

$h'(t) = \mathbf{A}h(t) + \mathbf{B}x(t) \\
y(t) = \mathbf{C}h(t)$


In [53]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim

Before, we had to create our own custom tokenizer, based on each char in our vocab_size. We will no longer do this! Let's use a pre-trained tokenizer from **HuggingFace** with all the fun bells and whistles.

In [54]:
from transformers import AutoTokenizer # HuggingFace 

# mamba uses "EleutherAI/gpt-neox-20b" tokenizer
tokenizer_name = "EleutherAI/gpt-neox-20b"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
test_input = "Hello world!"
tensor_input = tokenizer(test_input, return_tensors="pt")
print(tensor_input["input_ids"].shape) # 1 x L
print(tokenizer.decode(tensor_input["input_ids"][0]))
print(f"Vocab size: {tokenizer.vocab_size}")

torch.Size([1, 3])
Hello world!
Vocab size: 50254


We will also test out the train-test split and get_batch standards in **HuggingFace**. Should be of good use to us long down the road when we want to quickly set these features up for our model of choice.

In [55]:
from datasets import Dataset

# random set of strings (data)
raw_data = {
    "text": [
        "PyTorch makes tensor math easy.",
        "Hugging Face standardizes the data pipelines.",
        "NanoGPT is a great learning resource.",
        "Attention is all you need."
    ]
}

hf_dataset = Dataset.from_dict(raw_data)
print(hf_dataset)

Dataset({
    features: ['text'],
    num_rows: 4
})


In [56]:
# train-test split
hf_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(hf_split)
print(hf_split['train']['text'])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 3
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1
    })
})
Column(['NanoGPT is a great learning resource.', 'Hugging Face standardizes the data pipelines.', 'PyTorch makes tensor math easy.'])


In [57]:
# getting a batch with collator (this time for causal LLM)
from transformers import DataCollatorForLanguageModeling # define get_batch
from torch.utils.data import DataLoader # use get_batch 

def tokenize(dataset : Dataset):
    encode = lambda dataset : tokenizer(dataset['text'], truncation=True)
    return dataset.map(encode, batched=True) # batched -> parallel processing

# HF uses the map function to add tokenized input to dataset (with map)
res = tokenize(hf_split['train']) 
print(res)
print(res['text'])
print(res['input_ids'])

Map: 100%|██████████| 3/3 [00:00<00:00, 635.37 examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 3
})
Column(['NanoGPT is a great learning resource.', 'Hugging Face standardizes the data pipelines.', 'PyTorch makes tensor math easy.'])
Column([[47, 4692, 40, 5736, 310, 247, 1270, 4715, 7741, 15], [46941, 3390, 28801, 2629, 4219, 253, 941, 44387, 15], [14819, 22097, 348, 2789, 13148, 14168, 3477, 15]])


In [66]:
# convert tokenized into tensors

hf_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
loader = DataLoader(res['input_ids'], collate_fn=hf_collator, batch_size=1, shuffle=True)
print(next(iter(loader)))

{'input_ids': tensor([[  47, 4692,   40, 5736,  310,  247, 1270, 4715, 7741,   15]]), 'labels': tensor([[  47, 4692,   40, 5736,  310,  247, 1270, 4715, 7741,   15]])}



(2) ZOH Discretization

$\bar{\mathbf{A}} = \exp(\Delta \mathbf{A}) \\
\bar{\mathbf{B}} = (\Delta \mathbf{A})^{-1} (\exp(\Delta \mathbf{A}) - \mathbf{I}) \cdot \Delta \mathbf{B}$

(3) Discrete SSM

$h_t = \bar{\mathbf{A}}h_{t-1} + \bar{\mathbf{B}}x_t \\
y_t = \mathbf{C}h_t$


(4) Selective Mechanism

$\mathbf{B}_t = \text{Linear}_B(x_t) \\
\mathbf{C}_t = \text{Linear}_C(x_t) \\
\Delta_t = \text{softplus}(\text{Linear}_\Delta(x_t)) \\
\bar{\mathbf{A}}_t, \bar{\mathbf{B}}_t = \text{discretize}(\Delta_t, \mathbf{A}, \mathbf{B}_t)$